# Roboflow Face Mask Dataset Setup (Google Colab)

This notebook downloads the Roboflow dataset and converts YOLO annotations into a face-crop classifier dataset for training (`mask` vs `no_mask`).

## 1) Install dependencies

In [ ]:
!pip -q install --upgrade pip
!pip -q install roboflow opencv-python

## 2) Mount Drive and define paths

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path

ROOT = Path('/content/drive/MyDrive/thesis_mask_training')
RAW_ROOT = ROOT / 'raw_roboflow'
CLS_ROOT = ROOT / 'dataset'

RAW_ROOT.mkdir(parents=True, exist_ok=True)
CLS_ROOT.mkdir(parents=True, exist_ok=True)

print('ROOT:', ROOT)
print('RAW_ROOT:', RAW_ROOT)
print('CLS_ROOT:', CLS_ROOT)

## 3) Set Roboflow details

From your dataset URL:
- Workspace: `yolov8-dataset`
- Project: `face-mask-0zmpv`

Update `VERSION` to the version number shown in Roboflow.

In [ ]:
import os

ROBOFLOW_API_KEY = os.environ.get('ROBOFLOW_API_KEY', '')
if not ROBOFLOW_API_KEY:
    print('Set ROBOFLOW_API_KEY in Colab secrets or os.environ before running download cell.')

WORKSPACE = 'yolov8-dataset'
PROJECT = 'face-mask-0zmpv'
VERSION = 1  # change if your dataset version differs
FORMAT = 'yolov8'

## 4) Download dataset from Roboflow

In [ ]:
from roboflow import Roboflow

rf = Roboflow(api_key=ROBOFLOW_API_KEY)
project = rf.workspace(WORKSPACE).project(PROJECT)
version = project.version(VERSION)
dataset = version.download(FORMAT, location=str(RAW_ROOT / f'{PROJECT}-v{VERSION}-{FORMAT}'))

print('Downloaded to:', dataset.location)

## 5) Inspect classes from `data.yaml`

In [ ]:
import yaml
from pathlib import Path

raw_dir = Path(dataset.location)
yaml_path = raw_dir / 'data.yaml'
config = yaml.safe_load(yaml_path.read_text(encoding='utf-8'))
class_names = config.get('names', [])

print('Class names from data.yaml:')
for i, name in enumerate(class_names):
    print(i, name)

## 6) Define class mapping to binary labels

Map source classes into classifier labels:
- `mask` (proper + improper + partial mask, if present)
- `no_mask`

Edit mapping below after checking `data.yaml` class names.

In [ ]:
# Example keyword mapping; adjust to your dataset classes
MASK_KEYWORDS = {'mask', 'with_mask', 'proper_mask', 'incorrect_mask', 'improper_mask', 'mask_weared_incorrect'}
NO_MASK_KEYWORDS = {'no_mask', 'without_mask', 'unmask'}

def map_label(name: str):
    n = str(name).strip().lower()
    if n in MASK_KEYWORDS:
        return 'mask'
    if n in NO_MASK_KEYWORDS:
        return 'no_mask'
    # fallback heuristic
    if 'without' in n or 'no_mask' in n or n == 'nomask':
        return 'no_mask'
    if 'mask' in n:
        return 'mask'
    return None

mapped = {i: map_label(name) for i, name in enumerate(class_names)}
print('Class index -> mapped label')
for i, v in mapped.items():
    print(i, class_names[i], '=>', v)

## 7) Convert YOLO boxes to face crops for classifier dataset

In [ ]:
import cv2
from collections import Counter

def yolo_to_xyxy(xc, yc, w, h, img_w, img_h):
    x1 = int((xc - w / 2.0) * img_w)
    y1 = int((yc - h / 2.0) * img_h)
    x2 = int((xc + w / 2.0) * img_w)
    y2 = int((yc + h / 2.0) * img_h)
    x1 = max(0, min(x1, img_w - 1))
    y1 = max(0, min(y1, img_h - 1))
    x2 = max(1, min(x2, img_w))
    y2 = max(1, min(y2, img_h))
    return x1, y1, x2, y2

def split_dirs(raw_base):
    candidates = []
    for split in ['train', 'valid', 'test']:
        images_dir = raw_base / split / 'images'
        labels_dir = raw_base / split / 'labels'
        if images_dir.exists() and labels_dir.exists():
            candidates.append((split, images_dir, labels_dir))
    return candidates

def out_split_name(raw_split):
    return 'val' if raw_split == 'valid' else raw_split

for split in ['train', 'val', 'test']:
    for cls in ['mask', 'no_mask']:
        (CLS_ROOT / split / cls).mkdir(parents=True, exist_ok=True)

stats = Counter()

for raw_split, images_dir, labels_dir in split_dirs(raw_dir):
    out_split = out_split_name(raw_split)
    image_paths = sorted(list(images_dir.glob('*.jpg')) + list(images_dir.glob('*.jpeg')) + list(images_dir.glob('*.png')))

    for image_path in image_paths:
        label_path = labels_dir / (image_path.stem + '.txt')
        if not label_path.exists():
            continue

        image = cv2.imread(str(image_path))
        if image is None:
            continue
        h, w = image.shape[:2]

        lines = [ln.strip() for ln in label_path.read_text(encoding='utf-8').splitlines() if ln.strip()]
        for idx, ln in enumerate(lines):
            parts = ln.split()
            if len(parts) != 5:
                continue

            class_idx = int(float(parts[0]))
            mapped_label = mapped.get(class_idx)
            if mapped_label not in {'mask', 'no_mask'}:
                stats['skipped_unknown_label'] += 1
                continue

            xc, yc, bw, bh = map(float, parts[1:])
            x1, y1, x2, y2 = yolo_to_xyxy(xc, yc, bw, bh, w, h)
            if (x2 - x1) < 16 or (y2 - y1) < 16:
                stats['skipped_too_small'] += 1
                continue

            crop = image[y1:y2, x1:x2]
            if crop.size == 0:
                stats['skipped_empty_crop'] += 1
                continue

            out_name = f'{image_path.stem}_{idx}.jpg'
            out_path = CLS_ROOT / out_split / mapped_label / out_name
            cv2.imwrite(str(out_path), crop)

            stats[f'kept_{out_split}_{mapped_label}'] += 1
            stats['kept_total'] += 1

print('Conversion complete.')
for k in sorted(stats.keys()):
    print(k, stats[k])

## 8) Verify final classifier dataset counts

In [ ]:
for split in ['train', 'val', 'test']:
    for cls in ['mask', 'no_mask']:
        p = CLS_ROOT / split / cls
        count = len(list(p.glob('*.jpg'))) + len(list(p.glob('*.jpeg'))) + len(list(p.glob('*.png')))
        print(f'{split:5s} {cls:8s}: {count}')

## Next step

After this setup, open and run:
- `docs/instructions/camera/face_mask_colab_training.ipynb`

It already expects `ROOT/dataset/{train,val,test}/{mask,no_mask}`.